In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_m5_sales_train_validation"
CALENDAR_TABLE = f"{SILVER_SCHEMA}.silver_m5_calendar"
TARGET_TABLE = f"{SILVER_SCHEMA}.silver_m5_sales_daily"

TRANSFORM_RUN_ID = f"silver_m5_sales_daily_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "09_silver_m5_sales_daily"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("SOURCE_TABLE:", SOURCE_TABLE)
print("CALENDAR_TABLE:", CALENDAR_TABLE)
print("TARGET_TABLE:", TARGET_TABLE)

bronze_sales_df = spark.table(SOURCE_TABLE)
silver_calendar_df = spark.table(CALENDAR_TABLE)

bronze_sales_row_count = bronze_sales_df.count()
calendar_row_count = silver_calendar_df.count()

d_columns = [col_name for col_name in bronze_sales_df.columns if col_name.startswith("d_")]

print("Bronze sales row count:", bronze_sales_row_count)
print("Calendar row count:", calendar_row_count)
print("Number of M5 daily columns:", len(d_columns))
print("First 5 d columns:", d_columns[:5])
print("Last 5 d columns:", d_columns[-5:])

display(bronze_sales_df.limit(5))
bronze_sales_df.printSchema()

In [0]:
def clean_string_col(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


expected_daily_sales_row_count = bronze_sales_row_count * len(d_columns)

print("Expected daily sales rows:", expected_daily_sales_row_count)

# Build Spark stack expression to unpivot d_1 ... d_1913 into rows.
stack_expression = "stack({0}, {1}) as (m5_day_id, units_sold)".format(
    len(d_columns),
    ", ".join([f"'{col_name}', `{col_name}`" for col_name in d_columns])
)

sales_unpivot_df = (
    bronze_sales_df
    .selectExpr(
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        stack_expression
    )
)

silver_sales_daily_df = (
    sales_unpivot_df
    .select(
        clean_string_col("id").alias("m5_series_id"),
        clean_string_col("item_id").alias("item_id"),
        clean_string_col("dept_id").alias("department_id"),
        clean_string_col("cat_id").alias("category_id"),
        clean_string_col("store_id").alias("store_id"),
        clean_string_col("state_id").alias("state_id"),
        F.col("m5_day_id"),
        F.col("units_sold").cast("int").alias("units_sold")
    )
    .join(
        silver_calendar_df
        .select(
            "m5_day_id",
            "calendar_date_id",
            "calendar_date",
            "wm_year_week",
            "weekday_name",
            "month_number",
            "calendar_year",
            "is_weekend",
            "is_event_day",
            "event_count",
            "is_snap_any_state"
        ),
        on="m5_day_id",
        how="left"
    )
    .withColumn(
        "sales_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("m5_series_id"),
                F.col("store_id"),
                F.col("item_id"),
                F.col("m5_day_id")
            ),
            256
        )
    )
    .withColumn("is_valid_units_sold", F.col("units_sold").isNotNull() & (F.col("units_sold") >= 0))
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

# Cache because the next validation checks reuse this large dataframe.
silver_daily_row_count = silver_sales_daily_df.count()

print("Silver daily sales row count:", silver_daily_row_count)

display(
    silver_sales_daily_df
    .select(
        "sales_record_id",
        "m5_series_id",
        "item_id",
        "department_id",
        "category_id",
        "store_id",
        "state_id",
        "m5_day_id",
        "calendar_date",
        "wm_year_week",
        "units_sold",
        "is_weekend",
        "is_event_day",
        "is_snap_any_state"
    )
    .limit(20)
)

silver_sales_daily_df.printSchema()

In [0]:
sales_daily_validation_rows = []

# 1. Row count preserved through wide-to-long transform
sales_daily_validation_rows.append({
    "validation_check": "row_count_matches_expected_unpivot_count",
    "expected_value": str(expected_daily_sales_row_count),
    "actual_value": str(silver_daily_row_count),
    "status": "PASS" if silver_daily_row_count == expected_daily_sales_row_count else "FAIL",
    "issue_detail": None if silver_daily_row_count == expected_daily_sales_row_count else "Unpivoted daily sales row count does not match expected bronze rows × d columns."
})

# 2. Check source wide table has unique M5 series id
duplicate_source_series_count = (
    bronze_sales_df
    .groupBy("id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

sales_daily_validation_rows.append({
    "validation_check": "source_m5_series_id_unique",
    "expected_value": "0 duplicate source series ids",
    "actual_value": str(duplicate_source_series_count),
    "status": "PASS" if duplicate_source_series_count == 0 else "FAIL",
    "issue_detail": None if duplicate_source_series_count == 0 else "Duplicate id values found in bronze M5 sales source."
})

# 3. Aggregate row-level quality checks in one pass over the large dataframe
quality_summary = (
    silver_sales_daily_df
    .agg(
        F.sum(
            F.when(
                F.col("m5_series_id").isNull()
                | F.col("item_id").isNull()
                | F.col("store_id").isNull()
                | F.col("state_id").isNull()
                | F.col("m5_day_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_key_count"),

        F.sum(
            F.when(
                F.col("units_sold").isNull()
                | (F.col("units_sold") < 0),
                1
            ).otherwise(0)
        ).alias("invalid_units_sold_count"),

        F.sum(
            F.when(
                F.col("calendar_date").isNull()
                | F.col("calendar_date_id").isNull()
                | F.col("wm_year_week").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_calendar_join_count"),

        F.sum(
            F.when(F.col("sales_record_id").isNull(), 1).otherwise(0)
        ).alias("null_sales_record_id_count"),

        F.sum(
            F.when(~F.col("state_id").isin("CA", "TX", "WI"), 1).otherwise(0)
        ).alias("invalid_state_count")
    )
    .first()
)

null_key_count = int(quality_summary["null_key_count"])
invalid_units_sold_count = int(quality_summary["invalid_units_sold_count"])
missing_calendar_join_count = int(quality_summary["missing_calendar_join_count"])
null_sales_record_id_count = int(quality_summary["null_sales_record_id_count"])
invalid_state_count = int(quality_summary["invalid_state_count"])

sales_daily_validation_rows.append({
    "validation_check": "business_keys_not_null",
    "expected_value": "0 null key rows",
    "actual_value": str(null_key_count),
    "status": "PASS" if null_key_count == 0 else "FAIL",
    "issue_detail": None if null_key_count == 0 else "Null values found in key sales columns."
})

sales_daily_validation_rows.append({
    "validation_check": "units_sold_non_negative",
    "expected_value": "0 invalid units_sold rows",
    "actual_value": str(invalid_units_sold_count),
    "status": "PASS" if invalid_units_sold_count == 0 else "FAIL",
    "issue_detail": None if invalid_units_sold_count == 0 else "units_sold has null or negative values."
})

sales_daily_validation_rows.append({
    "validation_check": "calendar_join_complete",
    "expected_value": "0 missing calendar joins",
    "actual_value": str(missing_calendar_join_count),
    "status": "PASS" if missing_calendar_join_count == 0 else "FAIL",
    "issue_detail": None if missing_calendar_join_count == 0 else "Some M5 day ids did not join to the Silver calendar."
})

sales_daily_validation_rows.append({
    "validation_check": "sales_record_id_not_null",
    "expected_value": "0 null sales_record_id rows",
    "actual_value": str(null_sales_record_id_count),
    "status": "PASS" if null_sales_record_id_count == 0 else "FAIL",
    "issue_detail": None if null_sales_record_id_count == 0 else "Null sales_record_id values found."
})

sales_daily_validation_rows.append({
    "validation_check": "state_id_valid",
    "expected_value": "Only CA, TX, WI",
    "actual_value": str(invalid_state_count),
    "status": "PASS" if invalid_state_count == 0 else "FAIL",
    "issue_detail": None if invalid_state_count == 0 else "Unexpected state_id values found."
})

sales_daily_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

sales_daily_validation_df = spark.createDataFrame(
    sales_daily_validation_rows,
    schema=sales_daily_validation_schema
)

display(sales_daily_validation_df.orderBy("status", "validation_check"))

fail_count = sales_daily_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver daily sales validation failures:", fail_count)

In [0]:
if fail_count == 0:
    (
        silver_sales_daily_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("calendar_year", "state_id")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"✅ Wrote Silver daily sales table: {TARGET_TABLE}")
else:
    raise ValueError("Silver daily sales validation failed. Fix issues before writing.")

In [0]:
written_sales_daily_df = spark.table(TARGET_TABLE)

written_row_count = written_sales_daily_df.count()

print("Written Silver daily sales row count:", written_row_count)

display(
    written_sales_daily_df
    .select(
        "sales_record_id",
        "m5_series_id",
        "item_id",
        "department_id",
        "category_id",
        "store_id",
        "state_id",
        "m5_day_id",
        "calendar_date",
        "calendar_year",
        "wm_year_week",
        "units_sold",
        "is_weekend",
        "is_event_day",
        "is_snap_any_state"
    )
    .limit(20)
)

if written_row_count == expected_daily_sales_row_count:
    print("✅ Read-back check passed: Silver daily sales table exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected daily sales row count.")

In [0]:
sales_daily_validation_results_df = (
    sales_daily_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("target_table", F.lit(TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    sales_daily_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

print("✅ Notebook 09 PASSED: silver_m5_sales_daily created and validated.")
print("Next notebook: 10_silver_retail_inventory")